# auto-translate-pdf on Colab (Transformers + GPU)

Runs the same German→English, layout-preserving PDF pipeline as the [auto-translate-pdf](https://github.com/materialcritic/auto-translate-pdf) repo, but on Colab's GPU instead of a Mac's Apple Silicon.

**Why this differs from the local version:** the repo's `translate_pdf.py` uses `mlx-lm`, which only runs on Apple Silicon (Metal). Colab gives you a Linux VM with an NVIDIA GPU, so this notebook swaps in the original `google/translategemma-4b-it` checkpoint via `transformers` + 4-bit `bitsandbytes` instead — same model family, same structured chat-template format, different `generate()` call. The PDF extraction/reflow/rendering code is unchanged; only `load_model()`/`translate()` are replaced.

**Before running:** Runtime → Change runtime type → **T4 GPU** (or better, if you have Colab Pro).

In [ ]:
!nvidia-smi

## 1. Install dependencies

`fonts-liberation` matters: the repo's font-selection code falls back to Liberation Serif when it doesn't find macOS's Times New Roman, which is exactly the case here.

In [ ]:
!pip install -q transformers accelerate bitsandbytes pymupdf
!apt-get -qq install -y fonts-liberation > /dev/null

## 2. Get the extraction/reflow/rendering code

Clones the repo just to reuse `translate_pdf.py`'s PDF pipeline -- `load_model`/`translate` get monkeypatched with a transformers-based version below, nothing else about the file changes.

In [ ]:
!git clone -q https://github.com/materialcritic/auto-translate-pdf.git
import sys
sys.path.insert(0, "/content/auto-translate-pdf")

## 3. Load TranslateGemma 4B in 4-bit, and patch it in

`process_pdf` (in `translate_pdf.py`) calls `load_model(model_name)` and `translate(model, tokenizer, german_text)` by name from its own module's globals -- reassigning `translate_pdf.load_model`/`translate_pdf.translate` before calling `process_pdf` is enough for it to pick these up, no need to edit the file itself.

The chat-template quirk carries over unchanged from the local setup: TranslateGemma's template takes a structured `{"type": "text", "source_lang_code": ..., "target_lang_code": ..., "text": ...}` message, not a free-form prompt -- see `translate_pdf.py`'s `translate()` docstring for the MLX version of the same thing.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import translate_pdf

MODEL_ID = "google/translategemma-4b-it"


def load_model_transformers(model_name=MODEL_ID):
    print(f"Loading {model_name} (4-bit) ...")
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
    )
    model = AutoModelForCausalLM.from_pretrained(
        model_name, quantization_config=bnb_config, device_map="auto",
    )
    return model, tokenizer


def translate_transformers(model, tokenizer, german_text):
    messages = [{
        "role": "user",
        "content": [{
            "type": "text",
            "source_lang_code": "de",
            "target_lang_code": "en",
            "text": german_text,
            "image": None,
        }],
    }]
    inputs = tokenizer.apply_chat_template(
        messages, add_generation_prompt=True, return_tensors="pt", return_dict=True,
    ).to(model.device)

    # Same issue as the MLX version's load_model(): the tokenizer's default
    # eos_token_id doesn't include <end_of_turn>, which is what this chat
    # template actually emits to end a response -- without adding it,
    # generation runs to max_new_tokens on every paragraph.
    eos_ids = [tokenizer.eos_token_id]
    end_of_turn_id = tokenizer.convert_tokens_to_ids("<end_of_turn>")
    if end_of_turn_id is not None and end_of_turn_id != tokenizer.unk_token_id:
        eos_ids.append(end_of_turn_id)

    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=1024,
            do_sample=True,
            temperature=0.3,
            eos_token_id=eos_ids,
            pad_token_id=tokenizer.pad_token_id or tokenizer.eos_token_id,
        )
    new_tokens = out[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()


translate_pdf.load_model = load_model_transformers
translate_pdf.translate = translate_transformers
print("Patched. translate_pdf.process_pdf will use the transformers backend.")

## 4. Upload a German PDF

In [ ]:
from google.colab import files
uploaded = files.upload()
input_path = next(iter(uploaded))
print("Uploaded:", input_path)

## 5. Translate

Same `process_pdf` as the local pipeline -- pass `page_range=[0, 1, 2, 3, 4]` (0-indexed) as a fourth positional argument to translate only the first few pages first, since a long document can take a while even on a T4.

In [ ]:
from translate_pdf import process_pdf

output_path = input_path.rsplit(".", 1)[0] + "_en.pdf"
process_pdf(input_path, output_path, MODEL_ID)
print("Saved:", output_path)

## 6. Download the result

In [ ]:
files.download(output_path)

## Notes

- **No Folder Action here.** That's macOS-only automation; this notebook is upload → run → download, one document at a time.
- **No persistent model cache across sessions.** A fresh Colab runtime re-downloads the ~8GB full-precision checkpoint (quantized to 4-bit only after loading) every time, unless you mount Google Drive and point `HF_HOME` at it -- see the optional cell below if you'll be doing this often.
- **Free-tier Colab sessions disconnect/time out**, especially on a long document. If a run gets cut off partway, just re-run from section 3 (the model reload) -- there's no incremental resume within a single `process_pdf` call the way the local Folder Action watcher resumes across separate files.

### Optional: cache the model on Google Drive (persists across sessions)

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os
os.environ["HF_HOME"] = "/content/drive/MyDrive/hf_cache"
# Run this BEFORE section 3's model load so the download lands in Drive.